# Crew Content Pipeline

A `Crew` groups specialized agents under shared coordination. Each `CrewMember` carries a role, a backstory, and a set of tools — the crew assigns tasks to members and manages execution order so you focus on what each agent should do, not how they hand off to one another.

**What you'll build:** A three-member crew (researcher, writer, editor) that takes a topic, gathers key facts, drafts a 500-word article, then polishes it for tone and clarity.

**Time:** ~25 min | **Difficulty:** Intermediate

## Prerequisites & Setup

You need an `OPENAI_API_KEY` environment variable set.

**What you'll learn:**
- Define a `Crew` with named `CrewMember` agents
- Give each member a role, a backstory, and focused instructions
- Create `Task` objects and assign them to specific members
- Run the crew sequentially and inspect each agent's output
- Pass the output of one task as context to the next

In [ ]:
!pip install synapsekit -q

In [ ]:
import os

# Set your API key before running
# os.environ["OPENAI_API_KEY"] = "sk-..."

## Step 1: Import and Configure

A single LLM instance can be shared across all crew members, or each member can have its own — useful when different roles benefit from different models.

In [ ]:
from __future__ import annotations
import asyncio

from synapsekit.agents import Crew, CrewMember, Task
from synapsekit.llms.openai import OpenAILLM
from synapsekit import LLMConfig

# A single LLM instance can be shared across all crew members, or each member
# can have its own — useful when different roles benefit from different models.
llm = OpenAILLM(
    model="gpt-4o-mini",
    config=LLMConfig(temperature=0.7),
)

## Step 2: Define Crew Members

Backstories ground the agent's persona and make outputs more consistent.

In [ ]:
# Backstories ground the agent's persona and make outputs more consistent.
researcher = CrewMember(
    name="researcher",
    role="Research Analyst",
    backstory=(
        "You are a thorough research analyst with a talent for distilling "
        "complex topics into clear, accurate bullet-point summaries. "
        "You always cite the key claims you make."
    ),
    llm=llm,
)

writer = CrewMember(
    name="writer",
    role="Content Writer",
    backstory=(
        "You are an engaging content writer who transforms research notes "
        "into well-structured, reader-friendly articles. "
        "You write in an active voice and keep paragraphs short."
    ),
    llm=llm,
)

editor = CrewMember(
    name="editor",
    role="Senior Editor",
    backstory=(
        "You are a senior editor with high standards for clarity and flow. "
        "You tighten prose, fix awkward phrasing, and ensure the article "
        "has a strong opening and a satisfying conclusion."
    ),
    llm=llm,
)

## Step 3: Define Tasks

Each `Task` has a description, is assigned to a specific crew member, and uses `output_key` / `context_keys` to wire outputs from one task into the next.

In [ ]:
def build_tasks(topic: str) -> list[Task]:
    research_task = Task(
        description=f"Research the topic: '{topic}'. Produce 5\u20137 key facts, statistics, or insights in bullet-point form. Keep each bullet under 30 words.",
        assigned_to="researcher",
        # output_key names the field in the shared context other tasks can read
        output_key="research_notes",
    )

    write_task = Task(
        description="Using the research_notes from the previous task, write a 500-word article aimed at a general audience. Include an introduction, three body paragraphs, and a conclusion.",
        assigned_to="writer",
        # context_keys lists which previous outputs this task should receive
        context_keys=["research_notes"],
        output_key="draft_article",
    )

    edit_task = Task(
        description="Edit the draft_article for clarity, tone, and flow. Fix grammatical issues, tighten verbose sentences, and ensure the article reads smoothly from start to finish.",
        assigned_to="editor",
        context_keys=["draft_article"],
        output_key="final_article",
    )

    return [research_task, write_task, edit_task]

## Step 4: Assemble and Run the Crew

With `sequential=True`, each task waits for the previous one to finish, and the output is automatically injected into the next task's context when `context_keys` is specified.

In [ ]:
async def run_crew(topic: str) -> str:
    tasks = build_tasks(topic)

    crew = Crew(
        name="content-crew",
        members=[researcher, writer, editor],
        tasks=tasks,
        # sequential=True means each task waits for the previous one to finish.
        # The output of each task is automatically injected into the next task's
        # context when context_keys is specified.
        sequential=True,
        verbose=True,   # Prints each agent's action to stdout
    )

    result = await crew.arun()

    # result.outputs is a dict keyed by output_key
    final = result.outputs["final_article"]
    print("\n--- FINAL ARTICLE ---")
    print(final)
    return final

## Complete Working Example

Run the full crew pipeline on a topic. The researcher gathers facts, the writer drafts an article, and the editor polishes it.

In [ ]:
async def main():
    topic = "The environmental impact of large language models"
    await run_crew(topic)

asyncio.run(main())

## Next Steps

- **[Supervisor Agent Routing](supervisor-routing.ipynb)** — dynamically choose which agent handles each query
- **[Parallel Agent Execution](parallel-agent-execution.ipynb)** — run independent agents with `asyncio.gather()`
- **[Agent Handoff Chains](handoff-chains.ipynb)** — pass enriched context through a linear pipeline